In [ ]:
import torch, torchvision
import torch.nn as nn, torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader

t = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor()])
trd = DataLoader(torchvision.datasets.MNIST('./data', True,  transform=t, download=True), 64, True)
ted = DataLoader(torchvision.datasets.MNIST('./data', False, transform=t, download=True), 64)

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,6,5), nn.ReLU(), nn.AvgPool2d(2),
            nn.Conv2d(6,16,5), nn.ReLU(), nn.AvgPool2d(2),
            nn.Flatten(), nn.Linear(400,120), nn.ReLU(),
            nn.Linear(120,84), nn.ReLU(), nn.Linear(84,10)
        )
    def forward(self, x):
        return self.net(x)

model = LeNet5()
crit = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for e in range(5):
    model.train()
    rl = 0
    for x, y in trd:
        opt.zero_grad()
        o = model(x)
        l = crit(o, y)
        l.backward()
        opt.step()
        rl += l.item()
    print(f"E{e+1} | Loss: {rl/len(trd):.4f}")

In [ ]:
model.eval()
c = n = 0
with torch.no_grad():
    for x, y in ted:
        o = model(x)
        c += (o.argmax(1)==y).sum().item()
        n += len(y)
print(f"Test Accuracy: {100*c/n:.2f}%")